---------------------------------------**_Тюнинг Нейронной сети_**------------------------------------------

In [1]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [4]:
# Импорт нашего конфига и пайплайна предобработки
import config
from preprocessing import preprocess_data

# Функция фиксации сидов для 100% воспроизводимости результатов
def set_seed(seed: int = config.RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

# Выбор вычислительного устройства (GPU/CUDA при наличии или CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство для вычислений: {device}")

# 3. Загрузка исходных датасетов из путей в config.py
df_train = pd.read_csv(config.TRAIN_PATH)
df_test = pd.read_csv(config.TEST_PATH)
test_ids = df_test[config.ID_COL]
# 4. Предобработка (заполнение пропусков, кодирование категорий)
df_train_proc, df_test_proc = preprocess_data(df_train, df_test)
# 5. Разделение на признаки X и целевую переменную y (логарифм цены log1p)
X_train = df_train_proc.drop(columns=[config.ID_COL, config.TARGET_COL])
y_train = np.log1p(df_train_proc[config.TARGET_COL])
X_test = df_test_proc.drop(columns=[config.ID_COL])
# 6. Кросс-валидация
kf = KFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)
print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер целевой переменной: {y_train.shape}")
print(f"Размер тестовой выборки:  {X_test.shape}")

Используемое устройство для вычислений: cpu
Размер обучающей выборки: (1460, 208)
Размер целевой переменной: (1460,)
Размер тестовой выборки:  (1459, 208)


In [5]:
import importlib
importlib.reload(config)


class ResNetBlock(nn.Module):

    def __init__(self, dim: int, dropout: float = 0.15):
        super(ResNetBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.Dropout(dropout)
        )
        self.activation = nn.GELU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(x + self.block(x))


class HouseTabularResNet(nn.Module):
    """
    Продвинутая нейросеть для табличных данных с Residual Connections, LayerNorm и GELU:
    1. Проекция признаков: Linear(input_dim -> hidden_dim) -> LayerNorm -> GELU
    2. Серия остаточных блоков: num_blocks x ResNetBlock
    3. Выходной слой (Head): Linear(hidden_dim -> 1) без активации
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = config.DL_TUNING_CONFIG["hidden_dim"],
        num_blocks: int = config.DL_TUNING_CONFIG["num_blocks"],
        dropout: float = config.DL_TUNING_CONFIG["dropout"]
    ):
        super(HouseTabularResNet, self).__init__()

        # Входная проекция: разворачивает 208 признаков в латентное пространство hidden_dim
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Стек остаточных блоков ResNet
        self.blocks = nn.ModuleList([
            ResNetBlock(dim=hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])

        # Финальная регрессионная голова: скрытый вектор -> 1 непрерывное число
        self.head = nn.Linear(hidden_dim, 1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)
# Тестовая инициализация и проверка размерностей:
sample_model = HouseTabularResNet(input_dim=X_train.shape[1]).to(device)
sample_tensor = torch.randn(5, X_train.shape[1]).to(device)
sample_out = sample_model(sample_tensor)
print(sample_model)
print(f"\nТестовый прогон через ResNet успешен! Размер выхода: {sample_out.shape} (должно быть [5, 1])")

HouseTabularResNet(
  (input_layer): Sequential(
    (0): Linear(in_features=208, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.15, inplace=False)
  )
  (blocks): ModuleList(
    (0-1): 2 x ResNetBlock(
      (block): Sequential(
        (0): Linear(in_features=128, out_features=128, bias=True)
        (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.15, inplace=False)
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (6): Dropout(p=0.15, inplace=False)
      )
      (activation): GELU(approximate='none')
    )
  )
  (head): Linear(in_features=128, out_features=1, bias=True)
)

Тестовый прогон через ResNet успешен! Размер выхода: torch.Size([5, 1]) (должно быть [5, 1])


In [6]:
from torch.optim.lr_scheduler import CosineAnnealingLR

# Гиперпараметры тюнинга
EPOCHS = config.DL_TUNING_CONFIG["epochs"]
BATCH_SIZE = config.DL_TUNING_CONFIG["batch_size"]
LR = config.DL_TUNING_CONFIG["lr"]
WEIGHT_DECAY = config.DL_TUNING_CONFIG["weight_decay"]
ETA_MIN = config.DL_TUNING_CONFIG["eta_min"]


# Массивы для сбора Out-Of-Fold предсказаний и теста
dl_tuned_fold_scores = []
oof_dl_tuned_preds = np.zeros(len(X_train))
test_dl_tuned_preds = np.zeros(len(X_test))

print(f"Запуск обучения Tabular ResNet (Эпох: {EPOCHS}, Батч: {BATCH_SIZE}, LR: {LR} -> {ETA_MIN})...\n")
# ВНЕШНИЙ ЦИКЛ: 5 фолдов кросс-валидации
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):

    # Разделение данных текущего фолда
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    # Масштабирование признаков X (fit только на train фолда!)
    scaler_x = StandardScaler()
    X_tr_scaled = scaler_x.fit_transform(X_tr)
    X_val_scaled = scaler_x.transform(X_val)
    X_test_scaled = scaler_x.transform(X_test)

    # Масштабирование таргета y
    scaler_y = StandardScaler()
    y_tr_scaled = scaler_y.fit_transform(y_tr.values.reshape(-1, 1))

    # Превращаем в тензоры PyTorch
    train_x_tensor = torch.tensor(X_tr_scaled, dtype=torch.float32)
    train_y_tensor = torch.tensor(y_tr_scaled, dtype=torch.float32)
    val_x_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
    test_x_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    train_dataset = TensorDataset(train_x_tensor, train_y_tensor)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    # Инициализация модели, функции потерь, оптимизатора и планировщика
    model = HouseTabularResNet(input_dim=X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Косинусный планировщик LR на все эпохи
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=ETA_MIN)

    # ВНУТРЕННИЙ ЦИКЛ: Обучение по эпохам
    for epoch in range(EPOCHS):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            loss.backward()

            # Gradient Clipping: ограничиваем максимальную норму градиента числом 1.0
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

        # Шаг планировщика LR делаем строго в конце эпохи:
        scheduler.step()

    # ВАЛИДАЦИЯ ФОЛДА:
    model.eval()
    with torch.no_grad():
        val_preds_scaled = model(val_x_tensor.to(device)).cpu().numpy()
        val_preds = scaler_y.inverse_transform(val_preds_scaled).flatten()

        # Предсказание на тесте текущей моделью фолда:
        test_preds_scaled = model(test_x_tensor.to(device)).cpu().numpy()
        test_preds = scaler_y.inverse_transform(test_preds_scaled).flatten()
        test_dl_tuned_preds += test_preds / kf.n_splits

    oof_dl_tuned_preds[val_idx] = val_preds
    fold_rmsle = root_mean_squared_error(y_val, val_preds)
    dl_tuned_fold_scores.append(fold_rmsle)

    print(f"Фолд {fold} | RMSLE: {fold_rmsle:.4f} | Финальный LR: {scheduler.get_last_lr()[0]:.6f}")
# =============================================================
# ИТОГОВЫЕ МЕТРИКИ КРОСС-ВАЛИДАЦИИ
# =============================================================
mean_rmsle = np.mean(dl_tuned_fold_scores)
std_rmsle = np.std(dl_tuned_fold_scores)
real_y_true = np.expm1(y_train)
real_y_pred = np.expm1(oof_dl_tuned_preds)
dl_mae = mean_absolute_error(real_y_true, real_y_pred)
dl_r2 = r2_score(real_y_true, real_y_pred)
print("\n" + "=" * 55)
print(f"Итоговый средний RMSLE (Tuned Tabular ResNet): {mean_rmsle:.4f} (+/- {std_rmsle:.4f})")
print(f"Средняя ошибка (MAE) в долларах:             ${dl_mae:,.2f}")
print(f"Коэффициент детерминации (R^2):               {dl_r2:.4f}")
print("=" * 55)

Запуск обучения Tabular ResNet (Эпох: 100, Батч: 32, LR: 0.001 -> 1e-05)...

Фолд 1 | RMSLE: 0.1602 | Финальный LR: 0.000010
Фолд 2 | RMSLE: 0.1543 | Финальный LR: 0.000010
Фолд 3 | RMSLE: 0.1541 | Финальный LR: 0.000010
Фолд 4 | RMSLE: 0.1578 | Финальный LR: 0.000010
Фолд 5 | RMSLE: 0.1399 | Финальный LR: 0.000010

Итоговый средний RMSLE (Tuned Tabular ResNet): 0.1533 (+/- 0.0071)
Средняя ошибка (MAE) в долларах:             $18,190.78
Коэффициент детерминации (R^2):               0.8148


In [8]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import minimize


# =============================================================
# ИНИЦИАЛИЗАЦИЯ ЛУЧШИХ МОДЕЛЕЙ ИЗ НАШЕГО ТЮНИНГА
# =============================================================


# Лучший CatBoost (из Optuna):
best_cat = CatBoostRegressor(
    depth=6,
    learning_rate=0.0489,
    iterations=900,
    l2_leaf_reg=1.3478,
    random_strength=3.2359,
    bagging_temperature=0.3140,
    random_state=config.RANDOM_STATE,
    verbose=0
)

# Лучший LightGBM (из Optuna):
best_lgbm = LGBMRegressor(
    num_leaves=16,
    learning_rate=0.0171,
    n_estimators=800,
    colsample_bytree=0.4700,
    subsample=0.6811,
    reg_alpha=0.0029,
    reg_lambda=0.0112,
    random_state=config.RANDOM_STATE,
    verbosity=-1,
    n_jobs= -1
)

# Лучший Random Forest (из RandomizedSearch):
best_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=25,
    min_samples_split=10,
    min_samples_leaf=1,
    max_features=0.3,
    random_state=config.RANDOM_STATE,
    n_jobs= -1
)

# =============================================================
# БЫСТРЫЙ СБОР OOF-ПРЕДСКАЗАНИЙ БУСТИНГОВ НА ТЕХ ЖЕ 5 ФОЛДАХ
# =============================================================
print("Обучаем лучшие бустинги на 5 фолдах для сбора OOF-предсказаний...")
oof_cat = np.zeros(len(X_train))
oof_lgbm = np.zeros(len(X_train))
oof_rf = np.zeros(len(X_train))

for train_idx, val_idx in kf.split(X_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    best_cat.fit(X_tr, y_tr)
    oof_cat[val_idx] = best_cat.predict(X_val)

    best_lgbm.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = best_lgbm.predict(X_val)

    best_rf.fit(X_tr, y_tr)
    oof_rf[val_idx] = best_rf.predict(X_val)

# Базовый ансамбль деревьев (веса 0.50, 0.35, 0.15)
oof_trees = 0.50 * oof_cat + 0.35 * oof_lgbm + 0.15 * oof_rf
score_trees = root_mean_squared_error(y_train, oof_trees)
print(f"RMSLE чистого ансамбля бустингов (Cat + LGBM + RF): {score_trees:.4f}")

# =============================================================
#  АВТОМАТИЧЕСКИЙ ПОДБОР ВЕСА НЕЙРОСЕТИ В БЛЕНДИНГЕ
# =============================================================


# Функция потерь: RMSLE между y_train и взвешенной смесью деревьев и нейросети
def blending_objective(weight_dl):
    w_dl = weight_dl[0]
    w_trees = 1.0 - w_dl
    blend_preds = w_trees * oof_trees + w_dl * oof_dl_tuned_preds
    return root_mean_squared_error(y_train, blend_preds)


# Ищем оптимальный вес нейросети от 0.0 до 0.30
res = minimize(blending_objective, x0=[0.05], bounds=[(0.0, 0.30)], method='L-BFGS-B')
best_dl_weight = res.x[0]
best_blend_score = res.fun
print("=" * 55)
print(f"Оптимальный вес нейросети в бленде: {best_dl_weight * 100:.1f}%")
print(f"Вес ансамбля бустингов:              {(1 - best_dl_weight) * 100:.1f}%")
print(f"RMSLE итогового бленда (Trees + DL):  {best_blend_score:.4f}")


# Сравнение с базовым ансамблем
if best_blend_score < score_trees:
    diff = score_trees - best_blend_score
    print(f"Добавление нейросети улучшило скор на {diff:.5f}!")
else:
    print("Нейросеть не смогла улучшить сильный ансамбль бустингов (оптимальный вес 0%).")
print("=" * 55)

Обучаем лучшие бустинги на 5 фолдах для сбора OOF-предсказаний...
RMSLE чистого ансамбля бустингов (Cat + LGBM + RF): 0.1248
Оптимальный вес нейросети в бленде: 6.2%
Вес ансамбля бустингов:              93.8%
RMSLE итогового бленда (Trees + DL):  0.1247
🎉 УСПЕХ! Добавление нейросети улучшило скор на 0.00014!
